# 01 · 블록 마스킹과 상보 마스크

Fast-dLLM v2의 학습 아이디어를 작은 토큰 목록으로 확인한다. 이 코드는 논문의 모델·학습 결과를 재현하지 않는 **toy reproduction**이다.

**학습 목표**: block padding과 상보 마스크가 유효 token 위치를 어떻게 나누어 학습 신호를 보완하는지 설명한다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행한다. 외부 패키지는 없으며 Python 표준 기능만 사용한다.

In [ ]:
# 대문자 상수로 특수 토큰을 구분해 일반 문자열과 혼동하지 않게 한다.
MASK = '[MASK]'
PAD = '[PAD]'

def pad_to_block(tokens, block_size):
    remainder = len(tokens) % block_size
    return tokens + ([PAD] * ((block_size - remainder) % block_size))

def complementary_views(block, masked_indices):
    chosen = set(masked_indices)
    valid = {i for i, token in enumerate(block) if token != PAD}
    first = [MASK if i in chosen else token for i, token in enumerate(block)]
    complement = valid - chosen
    second = [MASK if i in complement else token for i, token in enumerate(block)]
    return first, second, chosen, complement


In [ ]:
tokens = 'OCR turns pixels into structured text'.split()
padded = pad_to_block(tokens, block_size=4)
blocks = [padded[i:i + 4] for i in range(0, len(padded), 4)]

for block in blocks:
    view_a, view_b, a, b = complementary_views(block, {0, 2})
    valid = {i for i, token in enumerate(block) if token != PAD}
    assert a.isdisjoint(b) and (a | b) >= valid
    print('원본 :', block)
    print('마스크:', view_a)
    print('상보 :', view_b, '\n')


상보 view를 함께 쓰면 유효 위치가 적어도 한쪽 학습 목표에 포함된다. 실제 논문은 shifted label, attention mask와 loss mask를 함께 사용하므로 이 목록 조작보다 훨씬 복잡하다.